In [ ]:
import pandas as pd
import polars as pl
import numpy as np
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
train = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
train.head(2)

# HOG

In [ ]:
y = train['label'].to_numpy()
X = train.drop('label', axis = 1).to_numpy().reshape(-1,28,28) # Переводим в формат (n_samples, 28,28)

In [ ]:
from skimage.feature import hog

# Вычисляем HOG признаки
hog_features = []
for img in tqdm(X,total = len(X), desc = 'Расчёт HOG признаков'):
    fd = hog(img, 
             orientations=9,      # количество направлений градиента
             pixels_per_cell=(7, 7),  # размер ячейки
             cells_per_block=(2, 2),  # блок для нормализации
             block_norm="L2-Hys")
    hog_features.append(fd)

hog_features = np.array(hog_features)

# Разделяем данные
X_train, X_valid, y_train, y_valid = train_test_split(hog_features, y, test_size=0.2, random_state=42)

# Обучаем простой классификатор
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Оценка
y_pred = clf.predict(X_valid)

print("Accuracy:", accuracy_score(y_valid, y_pred))
print(classification_report(y_valid, y_pred))

In [ ]:
test = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')
print(f'{test.shape = }')
test.head(2)

In [ ]:
X = test.to_numpy().reshape(-1,28,28) # Переводим в формат (n_samples, 28,28)
# Вычисляем HOG признаки
hog_features = []
for img in tqdm(X,total = len(X), desc = 'Расчёт HOG признаков'):
    fd = hog(img, 
             orientations=9,      # количество направлений градиента
             pixels_per_cell=(7, 7),  # размер ячейки
             cells_per_block=(2, 2),  # блок для нормализации
             block_norm="L2-Hys")
    hog_features.append(fd)

X_test = np.array(hog_features)

# Submission
y_pred = clf.predict(X_test)



In [ ]:
sample_submission = pd.read_csv('/kaggle/input/digit-recognizer/sample_submission.csv')
submission = pd.DataFrame({'ImageId': sample_submission['ImageId'], 'Label': y_pred})
submission.head(2)

In [ ]:
submission.to_csv("submission.csv", index=False)
print("submission.csv created!")